# 💼 ¿En qué trabaja España?
# EPA (INE) - ocupados por sector

#### 1️⃣ Cargamos y preparamos

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ocupados_rama.csv',
  sep='\t')

# Solo valores absolutos (no %)
df = df[
  df['Tipo de dato'] == 'Valor absoluto']

# Miles de personas: '22.221,1'
df['Total'] = pd.to_numeric(
  df['Total']
  .str.replace('.', '', regex=False)
  .str.replace(',', '.', regex=False),
  errors='coerce')

col = 'Rama de actividad - CNAE2009'

#### 2️⃣ Nos quedamos con las
#### secciones y les ponemos
#### nombre corto

Los datos son jerárquicos: la letra
es el sector, el número es su detalle.
Nos quedamos SOLO con las secciones.

In [ ]:
nombres = {
  'A': 'Agricultura', 'B': 'Minería',
  'C': 'Fábricas', 'D': 'Energía',
  'E': 'Agua y residuos',
  'F': 'Construcción', 'G': 'Comercio',
  'H': 'Transporte', 'I': 'Hostelería',
  'J': 'Comunicaciones',
  'K': 'Banca y seguros',
  'L': 'Inmobiliario',
  'M': 'Serv. profesionales',
  'N': 'Servicios a empresas',
  'O': 'Sector público',
  'P': 'Educación', 'Q': 'Sanidad',
  'R': 'Arte y ocio',
  'S': 'Otros servicios',
  'T': 'Empleo doméstico',
  'U': 'Organismos int.'}

# Qué incluye cada sector (para leerlo)
ejemplos = {
  'Comercio': 'Tiendas, súper, talleres',
  'Fábricas': 'Comida, coches, química',
  'Sanidad': 'Hospitales, médicos',
  'Hostelería': 'Bares, hoteles',
  'Educación': 'Colegios, universidades',
  'Construcción': 'Obras, reformas',
  'Sector público': 'Funcionarios, ejército',
  'Serv. profesionales':
    'Abogados, ingenieros',
  'Transporte': 'Camiones, mensajería',
  'Servicios a empresas':
    'ETT, seguridad, limpieza'}

# Secciones = letra + espacio
sec = df[df[col].str.match(r'^[A-U] ')].copy()
sec['sector'] = sec[col].str[0].map(nombres)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuáles son los sectores
### que MÁS gente emplean?

In [ ]:
# Total nacional de ocupados (miles)
total_pais = df[
  (df[col] == 'Total')
  & (df['Sexo'] == 'Ambos sexos')
  & (df['Periodo'] == 2025)
]['Total'].values[0]

top = (
  sec[
    (sec['Sexo'] == 'Ambos sexos')
    & (sec['Periodo'] == 2025)]
  .set_index('sector')['Total']
  .sort_values(ascending=False)
)

tabla = pd.DataFrame({
  'millones': (top / 1000).round(1),
  '%': (top / total_pais * 100).round(1),
})
tabla['incluye'] = tabla.index.map(ejemplos)

print(f"Total ocupados: {total_pais/1000:.1f}M")
print(tabla.head(10))

### 📈 ¿Y cómo han evolucionado
### estos sectores? (2008-2025)

In [ ]:
import matplotlib.pyplot as plt

# Serie por año y sector (millones)
evol = (
  sec[sec['Sexo'] == 'Ambos sexos']
  .pivot_table(
    index='Periodo',
    columns='sector',
    values='Total')
  / 1000
)

# Los 6 sectores más grandes hoy
top6 = (
  evol.iloc[-1]
  .sort_values(ascending=False)
  .head(6).index
)

evol[top6].plot(
  figsize=(9, 9),
  linewidth=3,
  marker='o')

plt.title('¿En qué trabaja España?',
  fontsize=20)
plt.ylabel('Millones de ocupados')
plt.xlabel('')
plt.legend(fontsize=13)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔍 PLOT TWIST:
### ¿Es España un país
### de servicios?

En el ranking la Industria salía 2ª.
Pero 'servicios' está troceado en
muchas secciones. Si agrupamos en
los 4 macrosectores, cambia todo.

In [ ]:
# A qué macrosector va cada sección
macro = {
  'A': 'Agricultura',
  'B': 'Industria', 'C': 'Industria',
  'D': 'Industria', 'E': 'Industria',
  'F': 'Construcción'}
# El resto (G-U) son Servicios

foto = sec[
  (sec['Sexo'] == 'Ambos sexos')
  & (sec['Periodo'] == 2025)].copy()

foto['macro'] = (
  foto[col].str[0]
  .map(lambda x: macro.get(x,
    'Servicios')))

r = (
  foto.groupby('macro')['Total'].sum()
  .sort_values(ascending=False)
)

resumen = pd.DataFrame({
  'millones': (r / 1000).round(1),
  '%': (r / r.sum() * 100).round(1),
})

print("España por macrosector:")
print(resumen)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué sectores son más
### de hombres y cuáles
### más de mujeres?

In [ ]:
foto = (
  sec[sec['Periodo'] == 2025]
  .pivot_table(
    index='sector',
    columns='Sexo',
    values='Total')
)

foto['%_mujeres'] = (
  foto['Mujeres']
  / foto['Ambos sexos'] * 100)

orden = foto['%_mujeres'].sort_values()

print("Más de HOMBRES (% mujeres):")
print(orden.head(3).round(0))
print("\nMás de MUJERES (% mujeres):")
print(orden.tail(3).round(0))

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué sector ha CRECIDO
### más y cuál ha caído
### en 10 años?

Ojo: quitamos sectores minúsculos,
un +150% en algo diminuto engaña.

In [ ]:
serie = (
  sec[sec['Sexo'] == 'Ambos sexos']
  .pivot_table(
    index='sector',
    columns='Periodo',
    values='Total')
)

# Solo sectores grandes (+300 mil)
grandes = serie[serie[2025] > 300]

crecimiento = (
  (grandes[2025] / grandes[2015] - 1)
  * 100
).sort_values(ascending=False)

print("Más creció 2015-2025 (%):")
print(crecimiento.head(3).round(0))
print("\nMás cayó (%):")
print(crecimiento.tail(3).round(0))